<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:22px 28px; border-radius:8px">
  <div style="font-size:0.9em; letter-spacing:2px; opacity:0.85">NOTEBOOK 06 · GOLD + PRE-SCREEN · COMPLETED</div>
  <div style="font-size:2.0em; font-weight:700; margin-top:4px">🏅 Gold layer: one audited cohort the coordinator can screen</div>
  <div style="font-size:1.1em; margin-top:8px; max-width:880px; opacity:0.95">
    Fuse the structured biomarkers with the NLP-recovered ones into a single source of truth,
    then compute trial eligibility with a plain-English reason for every patient, generically.
  </div>
</div>

## The gold layer is the answer the coordinator actually reads

| Gold table | What it is |
|---|---|
| `gold_unified_biomarker_profile` | One HER2/ER/PR row per patient, structured **or** NLP-recovered, with an audit column |
| `gold_trial_prescreen` | Per-patient, per-trial eligibility + a human-readable reason (LONG shape) |
| `gold_trial_prescreen_wide` | Backward-compatible one-row-per-patient view for the app |
| `gold_patient_measurements` | Per-patient test timeline for the app drill-down |

### The COALESCE-with-audit-column pattern (the heart of this notebook)

Two sources of biomarker truth: structured `silver_biomarker_profile` and the notes read by
`ai_query` (`silver_nlp_biomarkers`). The rule is simple and defensible:

1. **Prefer the structured value**, it's the lab's recorded result.
2. **Fall back to the NLP value** when no structured row exists (the notes-only patients).
3. **Record which source we used** in a `biomarker_source` column.

That audit column is what makes this trustworthy in a clinical setting. The AI never silently
overwrites a lab value, and every notes-derived decision is flagged for human review.

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## 0️⃣ Trials are DATA, not hardcoded rules (COMPLETED, our own `silver_trial_criteria`)

The pre-screen joins against a **trials catalog**, one row per trial, a `req_*` column per
criterion, `NULL` meaning "this trial does not constrain that field". Adding a trial becomes a
data change, not a code change. We seed a small catalog here so this notebook is self-contained.

<div style="background:#E3F2FD; border-left:6px solid #1565C0; padding:12px 16px; border-radius:4px">
<b>Trials are data:</b> because the join is generic, adding a trial is inserting a row into
<code>silver_trial_criteria</code>, not changing SQL. Trial C (triple-negative) is one of those
rows, screened by the same rule with zero code changes.
</div>

In [0]:
%sql
CREATE OR REPLACE TABLE silver_trial_criteria
COMMENT 'Trials-as-data catalog: one row per trial, one req_* column per criterion. NULL req_* = unconstrained.'
AS
SELECT * FROM VALUES
  -- trial_id, trial_name, status, req_sex, age_min, age_max, req_her2, req_er, req_pr,
  --   req_menopausal, req_no_prior_anti_her2, min_ecog, eligibility_text
  ('A', 'HER2-Positive Breast Cancer Study', 'Recruiting', 'Female', 18, 75,
   'Positive', CAST(NULL AS STRING), CAST(NULL AS STRING),
   CAST(NULL AS STRING), true, 1,
   'HER2-positive breast cancer, age 18-75, no prior anti-HER2 therapy.'),
  ('B', 'ER-Positive / HER2-Negative Study', 'Recruiting', 'Female', 18, 75,
   'Negative', 'Positive', CAST(NULL AS STRING),
   'Postmenopausal', CAST(NULL AS BOOLEAN), CAST(NULL AS INT),
   'ER-positive, HER2-negative, postmenopausal breast cancer, age 18-75.'),
  ('C', 'Triple-Negative Breast Cancer Study', 'Recruiting', 'Female', 18, 75,
   'Negative', 'Negative', 'Negative',
   CAST(NULL AS STRING), CAST(NULL AS BOOLEAN), CAST(NULL AS INT),
   'Triple-negative breast cancer (HER2-, ER-, PR-), age 18-75.')
AS t(trial_id, trial_name, status, req_sex, age_min, age_max, req_her2, req_er, req_pr,
      req_menopausal, req_no_prior_anti_her2, min_ecog, eligibility_text);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT trial_id, trial_name, req_sex, age_min, age_max, req_her2, req_er, req_pr, req_menopausal, req_no_prior_anti_her2, min_ecog
FROM silver_trial_criteria ORDER BY trial_id;

trial_id,trial_name,req_sex,age_min,age_max,req_her2,req_er,req_pr,req_menopausal,req_no_prior_anti_her2,min_ecog
A,HER2-Positive Breast Cancer Study,Female,18,75,Positive,null,null,null,true,1
B,ER-Positive / HER2-Negative Study,Female,18,75,Negative,Positive,null,Postmenopausal,null,null
C,Triple-Negative Breast Cancer Study,Female,18,75,Negative,Negative,Negative,null,null,null


## 1️⃣ `gold_unified_biomarker_profile`: fuse structured + NLP, keep the audit trail (COMPLETED)

A **FULL OUTER JOIN** on `person_id` keeps every patient in *either* source. For each marker,
`COALESCE(structured, nlp)`, structured wins, NLP fills the gap. `biomarker_source` is
`'structured'` whenever a structured row existed, else `'nlp'`.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_unified_biomarker_profile
COMMENT 'One HER2/ER/PR row per patient, structured preferred and NLP filling the gap, with a biomarker_source audit column.'
AS
SELECT
  COALESCE(s.person_id, n.person_id) AS person_id,
  COALESCE(s.her2_status, n.her2_status) AS her2_status,   -- structured wins
  COALESCE(s.er_status,   n.er_status)   AS er_status,
  COALESCE(s.pr_status,   n.pr_status)   AS pr_status,
  CASE WHEN s.person_id IS NOT NULL THEN 'structured' ELSE 'nlp' END AS biomarker_source
FROM silver_biomarker_profile s
FULL OUTER JOIN silver_nlp_biomarkers n ON s.person_id = n.person_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT biomarker_source, COUNT(*) AS patients
FROM gold_unified_biomarker_profile GROUP BY biomarker_source ORDER BY biomarker_source;

biomarker_source,patients
nlp,60
structured,240


## 2️⃣ `gold_trial_prescreen`: generic, catalog-driven eligibility + a reason (COMPLETED)

**The generic rule.** A patient qualifies for a trial when, for every `req_*` criterion that is
NON-NULL, the patient's value matches, AND `age_at_dx_years` is BETWEEN `age_min` AND `age_max`.
A NULL `req_*` column means the trial does not constrain that field. One rule, every trial.

**Shape: LONG (one row per person per trial).** The long shape
`(person_id, trial_id, eligible, reason, ...)` never changes as trials come and go.

| Catalog `req_*` column | Patient field it matches |
|---|---|
| `req_sex` | `gender` (case-insensitive: raw OMOP stores `FEMALE`) |
| `req_her2` / `req_er` / `req_pr` | `her2_status` / `er_status` / `pr_status` |
| `req_menopausal` | `menopausal_status` |
| `req_no_prior_anti_her2 = true` | patient's `prior_anti_her2` must be `false` |

`min_ecog` is carried through for the app to show, but not matched (patients have no ECOG field).

In [0]:
%sql
CREATE OR REPLACE TABLE gold_trial_prescreen
COMMENT 'Per-person per-trial eligibility via a generic join to silver_trial_criteria (NULL req_* = unconstrained). LONG shape. biomarker_source + min_ecog carried through.'
AS
WITH patient AS (
  -- One row per patient with every comparable field. menopausal_status / ajcc_stage / gender come
  -- from silver_demographics; prior_anti_her2 from silver_prior_therapy (NULL -> false).
  SELECT b.person_id,
         d.gender_source_value AS gender,
         d.age_at_dx_years,
         b.her2_status, b.er_status, b.pr_status,
         d.menopausal_status, d.ajcc_stage,
         b.biomarker_source,
         COALESCE(t.had_anti_her2_therapy, false) AS prior_anti_her2
  FROM gold_unified_biomarker_profile b
  INNER JOIN silver_demographics d ON b.person_id = d.person_id
  LEFT JOIN silver_prior_therapy t ON b.person_id = t.person_id
),
evaluated AS (
  SELECT p.*, c.trial_id, c.trial_name, c.status AS trial_status, c.eligibility_text, c.min_ecog,
         c.age_min, c.age_max, c.req_sex, c.req_her2, c.req_er, c.req_pr, c.req_menopausal,
         c.req_no_prior_anti_her2,
         (c.req_sex        IS NULL OR UPPER(p.gender)     = UPPER(c.req_sex)) AS ok_sex,
         (c.req_her2       IS NULL OR p.her2_status       = c.req_her2)       AS ok_her2,
         (c.req_er         IS NULL OR p.er_status         = c.req_er)         AS ok_er,
         (c.req_pr         IS NULL OR p.pr_status         = c.req_pr)         AS ok_pr,
         (c.req_menopausal IS NULL OR p.menopausal_status = c.req_menopausal) AS ok_menopausal,
         (c.req_no_prior_anti_her2 IS NULL
            OR (c.req_no_prior_anti_her2 = true AND p.prior_anti_her2 = false)) AS ok_prior_anti_her2,
         (p.age_at_dx_years BETWEEN c.age_min AND c.age_max) AS ok_age
  FROM patient p CROSS JOIN silver_trial_criteria c
)
SELECT person_id, trial_id, trial_name, trial_status, gender, age_at_dx_years,
       her2_status, er_status, pr_status, menopausal_status, ajcc_stage,
       prior_anti_her2, biomarker_source, min_ecog, eligibility_text,

  -- eligible: AND of every criterion. A NULL ok_* (a required value the patient is missing)
  -- counts as not-eligible, so we COALESCE each flag to false.
  ( COALESCE(ok_sex, false) AND COALESCE(ok_her2, false) AND COALESCE(ok_er, false)
    AND COALESCE(ok_pr, false) AND COALESCE(ok_menopausal, false)
    AND COALESCE(ok_prior_anti_her2, false) AND COALESCE(ok_age, false) ) AS eligible,

  -- reason: when eligible, summarize the constrained biomarkers with the source; else name the
  -- FIRST failing criterion. Precedence: biomarkers, menopausal, age, prior therapy, sex.
  CASE
    WHEN ( COALESCE(ok_sex, false) AND COALESCE(ok_her2, false) AND COALESCE(ok_er, false)
           AND COALESCE(ok_pr, false) AND COALESCE(ok_menopausal, false)
           AND COALESCE(ok_prior_anti_her2, false) AND COALESCE(ok_age, false) )
      THEN concat('Eligible: ',
             concat_ws(', ',
               CASE WHEN req_her2       IS NOT NULL THEN concat('HER2 ', her2_status) END,
               CASE WHEN req_er         IS NOT NULL THEN concat('ER ', er_status) END,
               CASE WHEN req_pr         IS NOT NULL THEN concat('PR ', pr_status) END,
               CASE WHEN req_menopausal IS NOT NULL THEN menopausal_status END,
               concat('age ', age_at_dx_years)),
             ' (', biomarker_source, ')')
    WHEN NOT COALESCE(ok_her2, false)       THEN concat('Excluded: HER2 is ', COALESCE(her2_status, 'Unknown'), ' (need ', req_her2, ')')
    WHEN NOT COALESCE(ok_er, false)         THEN concat('Excluded: ER is ',   COALESCE(er_status,   'Unknown'), ' (need ', req_er, ')')
    WHEN NOT COALESCE(ok_pr, false)         THEN concat('Excluded: PR is ',   COALESCE(pr_status,   'Unknown'), ' (need ', req_pr, ')')
    WHEN NOT COALESCE(ok_menopausal, false) THEN concat('Excluded: ', COALESCE(menopausal_status, 'menopausal status unknown'), ' (need ', req_menopausal, ')')
    WHEN NOT COALESCE(ok_age, false)        THEN concat('Excluded: age ', age_at_dx_years, ' outside ', age_min, ' to ', age_max)
    WHEN NOT COALESCE(ok_prior_anti_her2, false) THEN 'Excluded: prior anti-HER2 therapy'
    WHEN NOT COALESCE(ok_sex, false)        THEN concat('Excluded: sex ', gender, ' (need ', req_sex, ')')
    ELSE 'Excluded'
  END AS reason
FROM evaluated;

num_affected_rows,num_inserted_rows


### Backward-compatible wide view (PRE-BUILT)

The app reads `trial_a_eligible` / `trial_b_eligible` / `trial_c_eligible`. We pivot the long
table back to a one-row-per-patient view for that path; new trials stay available in the long table.

In [0]:
%sql
CREATE OR REPLACE VIEW gold_trial_prescreen_wide
COMMENT 'Backward-compatible wide view: one row per patient with trial_a_/trial_b_/trial_c_ columns, from the long table.'
AS
SELECT
  person_id,
  FIRST(gender)            AS gender,
  FIRST(age_at_dx_years)   AS age_at_dx_years,
  FIRST(her2_status)       AS her2_status,
  FIRST(er_status)         AS er_status,
  FIRST(pr_status)         AS pr_status,
  FIRST(menopausal_status) AS menopausal_status,
  FIRST(ajcc_stage)        AS ajcc_stage,
  FIRST(prior_anti_her2)   AS prior_anti_her2,
  FIRST(biomarker_source)  AS biomarker_source,
  MAX(CASE WHEN trial_id = 'A' THEN eligible END) AS trial_a_eligible,
  MAX(CASE WHEN trial_id = 'A' THEN reason   END) AS trial_a_reason,
  MAX(CASE WHEN trial_id = 'B' THEN eligible END) AS trial_b_eligible,
  MAX(CASE WHEN trial_id = 'B' THEN reason   END) AS trial_b_reason,
  MAX(CASE WHEN trial_id = 'C' THEN eligible END) AS trial_c_eligible,
  MAX(CASE WHEN trial_id = 'C' THEN reason   END) AS trial_c_reason
FROM gold_trial_prescreen
GROUP BY person_id;

In [0]:
%sql
SELECT person_id, trial_id, trial_name, her2_status, er_status, menopausal_status,
       age_at_dx_years, biomarker_source, eligible, reason
FROM gold_trial_prescreen ORDER BY person_id, trial_id LIMIT 15;

person_id,trial_id,trial_name,her2_status,er_status,menopausal_status,age_at_dx_years,biomarker_source,eligible,reason
1,A,HER2-Positive Breast Cancer Study,Positive,Positive,null,35,structured,true,"Eligible: HER2 Positive, age 35 (structured)"
1,B,ER-Positive / HER2-Negative Study,Positive,Positive,null,35,structured,false,Excluded: HER2 is Positive (need Negative)
1,C,Triple-Negative Breast Cancer Study,Positive,Positive,null,35,structured,false,Excluded: HER2 is Positive (need Negative)
2,A,HER2-Positive Breast Cancer Study,Positive,Negative,null,38,structured,true,"Eligible: HER2 Positive, age 38 (structured)"
2,B,ER-Positive / HER2-Negative Study,Positive,Negative,null,38,structured,false,Excluded: HER2 is Positive (need Negative)
2,C,Triple-Negative Breast Cancer Study,Positive,Negative,null,38,structured,false,Excluded: HER2 is Positive (need Negative)
3,A,HER2-Positive Breast Cancer Study,Positive,Negative,Postmenopausal,58,structured,true,"Eligible: HER2 Positive, age 58 (structured)"
3,B,ER-Positive / HER2-Negative Study,Positive,Negative,Postmenopausal,58,structured,false,Excluded: HER2 is Positive (need Negative)
3,C,Triple-Negative Breast Cancer Study,Positive,Negative,Postmenopausal,58,structured,false,Excluded: HER2 is Positive (need Negative)
4,A,HER2-Positive Breast Cancer Study,Positive,Positive,null,29,structured,true,"Eligible: HER2 Positive, age 29 (structured)"


## 3️⃣ The payoff: the unified cohort is *bigger* than structured-only 🎯 (PRE-BUILT)

If the coordinator had only the structured pipeline, every eligible patient tagged
`biomarker_source = 'nlp'` would be **invisible**. Count eligibility split by source.

In [0]:
%sql
SELECT biomarker_source,
  SUM(CASE WHEN trial_id = 'A' AND eligible THEN 1 ELSE 0 END) AS trial_a_eligible,
  SUM(CASE WHEN trial_id = 'B' AND eligible THEN 1 ELSE 0 END) AS trial_b_eligible,
  SUM(CASE WHEN trial_id = 'C' AND eligible THEN 1 ELSE 0 END) AS trial_c_eligible
FROM gold_trial_prescreen GROUP BY biomarker_source ORDER BY biomarker_source;

biomarker_source,trial_a_eligible,trial_b_eligible,trial_c_eligible
nlp,31,14,13
structured,109,56,40


In [0]:
nlp = spark.sql("""
  SELECT
    SUM(CASE WHEN trial_id = 'A' AND eligible AND biomarker_source = 'nlp' THEN 1 ELSE 0 END) AS a_nlp,
    SUM(CASE WHEN trial_id = 'B' AND eligible AND biomarker_source = 'nlp' THEN 1 ELSE 0 END) AS b_nlp,
    SUM(CASE WHEN trial_id = 'A' AND eligible THEN 1 ELSE 0 END)                              AS a_total,
    SUM(CASE WHEN trial_id = 'B' AND eligible THEN 1 ELSE 0 END)                              AS b_total
  FROM gold_trial_prescreen
""").first()

show_md(f"""
<div style="background:#FFEBEE; border-left:6px solid #C8102E; padding:12px 16px; border-radius:4px">
<b>Structured SQL alone would have missed {nlp['a_nlp'] + nlp['b_nlp']} eligible patients.</b><br>
Trial A: {nlp['a_nlp']} of {nlp['a_total']} eligible patients were recovered only from the notes.<br>
Trial B: {nlp['b_nlp']} of {nlp['b_total']} eligible patients were recovered only from the notes.
</div>
<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px; margin-top:10px">
Those <b>{nlp['a_nlp'] + nlp['b_nlp']} patients</b> are now in the unified cohort, each flagged
<code>biomarker_source = 'nlp'</code> so a coordinator knows to confirm the note before enrolling.
</div>
""")

Structured SQL alone would have missed 45 eligible patients. 
Trial A: 31 of 140 eligible patients were recovered only from the notes. 
Trial B: 14 of 70 eligible patients were recovered only from the notes.
 
 
Those 45 patients are now in the unified cohort, each flagged
 biomarker_source = 'nlp' so a coordinator knows to confirm the note before enrolling.

In [0]:
%sql
-- Trials A and B carry catalog criteria identical to the old hardcoded rules, so the generic join
-- MUST reproduce Trial A = 140, Trial B = 70, +31 NLP-recovered for A and +14 for B. Trial C is net-new.
SELECT trial_id,
       SUM(CASE WHEN eligible THEN 1 ELSE 0 END) AS eligible_patients,
       SUM(CASE WHEN eligible AND biomarker_source = 'nlp' THEN 1 ELSE 0 END) AS nlp_recovered
FROM gold_trial_prescreen GROUP BY trial_id ORDER BY trial_id;

trial_id,eligible_patients,nlp_recovered
A,140,31
B,70,14
C,53,13


## 5️⃣ `gold_patient_measurements`: a per-patient test timeline for the app (PRE-BUILT)

The coordinator app interrogates one patient: "what tests did this patient have, and when?" This
longitudinal view is built straight from the raw OMOP `measurement` table, one row per result.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_patient_measurements
COMMENT 'Longitudinal per-patient test timeline for the coordinator app drill-down. One row per measurement, from the raw OMOP measurement table.'
AS
SELECT
  person_id,
  COALESCE(measurement_date, CAST(measurement_datetime AS DATE)) AS measurement_date,
  measurement_source_value                                       AS test_name,
  COALESCE(value_source_value, CAST(value_as_number AS STRING))  AS value,
  unit_source_value                                              AS unit
FROM measurement
WHERE person_id IS NOT NULL
ORDER BY person_id, measurement_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT person_id, measurement_date, test_name, value, unit
FROM gold_patient_measurements ORDER BY person_id, measurement_date LIMIT 20;

person_id,measurement_date,test_name,value,unit
1,2023-01-11,Estrogen receptor,Positive,null
1,2023-01-11,HER2/neu,Positive,null
1,2023-01-11,Progesterone receptor,Negative,null
2,2024-05-08,Progesterone receptor,Positive,null
2,2024-05-08,HER2/neu,Positive,null
2,2024-05-08,Estrogen receptor,Negative,null
3,2024-11-30,Progesterone receptor,Negative,null
3,2024-11-30,HER2/neu,Positive,null
3,2024-11-30,Estrogen receptor,Negative,null
4,2024-04-28,HER2/neu,Positive,null


## 🚩 Checkpoint 6: gold cohort is catalog-driven and app-ready

- `gold_trial_prescreen` is **LONG** (one row per person per trial), computed by a single generic
  join to `silver_trial_criteria`. Adding a trial is a data change.
- Trials A and B reproduce the validated numbers (**A = 140**, **B = 70**, **+31 NLP-recovered for A, +14 for B**);
  Trial C (triple-negative) is net-new.
- `gold_trial_prescreen_wide` keeps the app's `trial_a_/trial_b_/trial_c_` shape.
- `gold_patient_measurements` gives the app a per-patient test timeline.

## ▶️ Next step
### → Open **[07_mlflow_evaluation_runs]($./07_mlflow_evaluation_runs)** to score the NLP extractor against ground truth with MLflow.